# Projet Deep Learning — Reconnaissance d'expressions faciales

L'idée : apprendre à un réseau de neurones à reconnaître l'émotion d'un visage (joie, colère, tristesse...).
On avance étape par étape : d'abord les données, puis un modèle simple, puis un CNN, et on améliore au fur et à mesure.

## Partie 1 — Les données

### 1.1 Installation et imports

Sur Colab, TensorFlow est déjà installé. Il nous manque juste `kagglehub` pour récupérer le dataset directement depuis Kaggle.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import kagglehub

print("TensorFlow :", tf.__version__)

### 1.2 Le dataset choisi : FER-2013

On part sur **FER-2013**, le dataset de référence pour ce genre de projet :
- **Source** : challenge Kaggle *"Challenges in Representation Learning: Facial Expression Recognition"* (ICML 2013), créé par Pierre-Luc Carrier et Aaron Courville. On utilise la version en images de [msambare sur Kaggle](https://www.kaggle.com/datasets/msambare/fer2013).
- **Licence** : usage pour la recherche / l'éducation (les images viennent de Google Images).
- **Contenu** : des visages déjà recadrés, en **48×48 pixels**, en **niveaux de gris**, au format **JPG**.
- **7 classes** : angry, disgust, fear, happy, neutral, sad, surprise.

Le dataset est déjà séparé en deux dossiers `train/` et `test/`, avec un sous-dossier par émotion. Pratique : le nom du dossier, c'est le label !

In [ ]:
# Téléchargement (la première fois ça prend quelques secondes, ensuite c'est en cache)
data_dir = kagglehub.dataset_download("msambare/fer2013")
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")

print("Dataset téléchargé dans :", data_dir)
print("Contenu :", os.listdir(data_dir))

### 1.3 Premier coup d'œil

On compte combien d'images il y a dans chaque classe, pour le train et le test.

In [ ]:
classes = sorted(os.listdir(train_dir))
print("Classes :", classes)

# On compte les fichiers dans chaque dossier
comptes = pd.DataFrame({
    "train": [len(os.listdir(os.path.join(train_dir, c))) for c in classes],
    "test": [len(os.listdir(os.path.join(test_dir, c))) for c in classes],
}, index=classes)
comptes.loc["TOTAL"] = comptes.sum()
comptes

Et on ouvre une image au hasard pour vérifier sa taille et son format :

In [ ]:
dossier = os.path.join(train_dir, "happy")
fichier = os.listdir(dossier)[0]
img = Image.open(os.path.join(dossier, fichier))

print("Fichier :", fichier)
print("Taille :", img.size, "| Mode :", img.mode, "(L = niveaux de gris)")

plt.imshow(img, cmap="gray")
plt.title("happy")
plt.axis("off")
plt.show()

### 1.4 Toutes les images ont-elles le même format ?

On a regardé une seule image, mais rien ne garantit que toutes sont pareilles. On vérifie sur tout le dataset.

In [ ]:
tailles = set()
modes = set()
extensions = set()

for dossier in [train_dir, test_dir]:
    for c in classes:
        for f in os.listdir(os.path.join(dossier, c)):
            img = Image.open(os.path.join(dossier, c, f))
            tailles.add(img.size)
            modes.add(img.mode)
            extensions.add(os.path.splitext(f)[1])

print("Tailles trouvées :", tailles)
print("Modes trouvés :", modes)
print("Extensions trouvées :", extensions)

### 1.5 Répartition des classes

On trace le nombre d'images par émotion. Si les barres sont très différentes, le dataset est **déséquilibré** : le modèle risque de favoriser les classes qui ont beaucoup d'exemples.

In [ ]:
comptes_classes = comptes.drop("TOTAL")

comptes_classes.plot(kind="bar", figsize=(9, 4))
plt.title("Nombre d'images par classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=0)
plt.show()

# En pourcentage, c'est plus parlant
pourcentages = (comptes_classes["train"] / comptes_classes["train"].sum() * 100).round(1)
print("Part de chaque classe dans le train (%) :")
print(pourcentages)
print()
print("Ratio classe la plus fréquente / la moins fréquente :",
      round(comptes_classes["train"].max() / comptes_classes["train"].min(), 1))

**Ce qu'on remarque :**
- `happy` écrase tout le monde (~25 % du train).
- `disgust` est quasi absente (~1.5 %) : environ 16 fois moins d'images que `happy`.
- Le test a la même répartition que le train, donc l'accuracy sur le test sera elle aussi tirée par `happy`.

Conséquence : un modèle qui répondrait "happy" à chaque fois aurait déjà ~25 % d'accuracy sans rien apprendre. C'est pour ça qu'on regardera la **matrice de confusion** et les scores **par classe**, et pas juste l'accuracy globale.

### 1.6 À quoi ressemblent les visages ?

On affiche quelques images au hasard pour chaque émotion (une ligne = une classe).

In [ ]:
nb_par_classe = 8
np.random.seed(0)

fig, axes = plt.subplots(len(classes), nb_par_classe, figsize=(nb_par_classe * 1.3, len(classes) * 1.5))

for i, c in enumerate(classes):
    fichiers = os.listdir(os.path.join(train_dir, c))
    choix = np.random.choice(fichiers, nb_par_classe, replace=False)
    for j, f in enumerate(choix):
        img = Image.open(os.path.join(train_dir, c, f))
        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

**Ce qu'on remarque :**
- Les visages sont recadrés et à peu près centrés, mais l'angle, l'éclairage et l'âge des personnes varient beaucoup.
- Certaines images sont floues, partiellement cachées (mains, lunettes de soleil), ou ont un filigrane par-dessus : FER-2013 est connu pour être un peu "bruité".
- Certaines émotions se ressemblent beaucoup (`fear` / `surprise`, `sad` / `neutral`, `angry` / `disgust`) : on peut s'attendre à ce que le modèle les confonde.

### 1.7 Préparation des données

Un réseau de neurones ne lit pas des fichiers JPG : il veut des **tableaux de nombres**. On va donc :
1. **Charger** toutes les images dans des tableaux numpy (`X` = les images, `y` = les labels).
2. **Redimensionner** : pas besoin ici, on a vérifié que tout est déjà en 48×48.
3. **Normaliser** les pixels.
4. **Encoder** les labels.
5. **Découper** en train / validation / test.

#### Chargement

Pour chaque image, on garde ses pixels dans `X` et le numéro de sa classe dans `y` (0 = angry, 1 = disgust, ..., 6 = surprise). Le numéro vient simplement de la position du dossier dans la liste `classes`.

In [ ]:
X_train_full, y_train_full = [], []
X_test, y_test = [], []

for i, c in enumerate(classes):
    for f in os.listdir(os.path.join(train_dir, c)):
        X_train_full.append(np.array(Image.open(os.path.join(train_dir, c, f))))
        y_train_full.append(i)
    for f in os.listdir(os.path.join(test_dir, c)):
        X_test.append(np.array(Image.open(os.path.join(test_dir, c, f))))
        y_test.append(i)

X_train_full = np.array(X_train_full)
y_train_full = np.array(y_train_full)
X_test = np.array(X_test)
y_test = np.array(y_test)

print("X_train_full :", X_train_full.shape, X_train_full.dtype)
print("y_train_full :", y_train_full.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

On a 28 709 images de 48×48 pour le train. Keras attend aussi une dimension pour les **canaux** (1 en niveaux de gris, 3 en couleur), donc on passe de `(N, 48, 48)` à `(N, 48, 48, 1)`.

In [ ]:
X_train_full = X_train_full.reshape(-1, 48, 48, 1)
X_test = X_test.reshape(-1, 48, 48, 1)

print("X_train_full :", X_train_full.shape)
print("X_test :", X_test.shape)

#### Normalisation

Les pixels vont de 0 (noir) à 255 (blanc). On les divise par 255 pour les ramener entre **0 et 1**.

Pourquoi ? Avec des valeurs trop grandes, les sommes pondérées dans les neurones explosent et les gradients aussi : l'entraînement devient instable et lent. Avec des entrées entre 0 et 1, tout reste à la même échelle et la descente de gradient se passe beaucoup mieux.

In [ ]:
# On garde une copie des valeurs brutes juste pour comparer
pixels_avant = X_train_full[:1000].flatten()

X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

pixels_apres = X_train_full[:1000].flatten()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(pixels_avant, bins=50)
axes[0].set_title("Pixels avant normalisation (0 → 255)")
axes[1].hist(pixels_apres, bins=50, color="tab:orange")
axes[1].set_title("Pixels après normalisation (0 → 1)")
plt.show()

print("Min / max après :", X_train_full.min(), "/", X_train_full.max())

La forme de l'histogramme ne change pas du tout, seule l'échelle de l'axe x change : on n'a perdu aucune information.

#### Découpage train / validation / test

On a déjà un jeu de **test** fourni par le dataset. On n'y touche plus jusqu'à l'évaluation finale : si on s'en servait pour régler le modèle, le score final serait trop optimiste.

Pour suivre l'entraînement et comparer nos modèles, on a besoin d'un jeu de **validation**. On le prend dans le train : 15 % des images, tirées au hasard.

On utilise `stratify=y_train_full` pour garder **la même répartition des classes** dans le train et la validation. Sans ça, avec seulement 436 images `disgust`, on pourrait se retrouver avec très peu de `disgust` en validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    stratify=y_train_full,
    random_state=42,
)

print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

On vérifie que la stratification a bien marché (les pourcentages doivent être quasi identiques) :

In [ ]:
repartition = pd.DataFrame({
    "train (%)": np.bincount(y_train) / len(y_train) * 100,
    "validation (%)": np.bincount(y_val) / len(y_val) * 100,
    "test (%)": np.bincount(y_test) / len(y_test) * 100,
}, index=classes).round(1)
repartition

#### Encodage des labels (one-hot)

Pour l'instant un label est un simple numéro, par exemple `3` pour happy. Mais notre réseau va sortir **7 probabilités** (une par classe, grâce au Softmax). Pour pouvoir comparer la sortie avec la vraie réponse, on transforme le label en vecteur de 7 cases avec un 1 à la bonne place :

`3` → `[0, 0, 0, 1, 0, 0, 0]`

C'est le **one-hot encoding**. Ça évite aussi que le réseau croie qu'il y a un ordre entre les classes (`surprise` = 6 n'est pas "plus grand" que `angry` = 0).

On garde aussi les versions en numéros (`y_train`, `y_val`, `y_test`), elles seront pratiques pour la matrice de confusion.

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_oh = to_categorical(y_train, num_classes=len(classes))
y_val_oh = to_categorical(y_val, num_classes=len(classes))
y_test_oh = to_categorical(y_test, num_classes=len(classes))

print("y_train :", y_train.shape, "→ y_train_oh :", y_train_oh.shape)

### 1.8 Vérification finale

Avant de passer aux modèles, on affiche quelques images **telles que le réseau va les recevoir** (déjà normalisées), avec leur label en numéro, en one-hot et en nom. Si tout colle, la préparation est bonne.

In [ ]:
np.random.seed(1)
indices = np.random.choice(len(X_train), 6, replace=False)

fig, axes = plt.subplots(1, 6, figsize=(15, 3.5))
for ax, idx in zip(axes, indices):
    ax.imshow(X_train[idx].squeeze(), cmap="gray")
    ax.set_title(f"{classes[y_train[idx]]} ({y_train[idx]})\n{y_train_oh[idx].astype(int)}", fontsize=9)
    ax.axis("off")
plt.show()

**Récap de ce qu'on a en sortie :**

| Jeu | Variable | Forme | Rôle |
|---|---|---|---|
| Train | `X_train`, `y_train_oh` | (24 402, 48, 48, 1) | apprendre les poids |
| Validation | `X_val`, `y_val_oh` | (4 307, 48, 48, 1) | suivre l'entraînement, comparer les modèles |
| Test | `X_test`, `y_test_oh` | (7 178, 48, 48, 1) | évaluation finale uniquement |

## Partie 2 — Modèle de référence (réseau dense)

Avant de faire un CNN, on construit un réseau "classique" tout simple. Il ne sera pas très bon, et c'est normal : il sert de **point de comparaison**. Si notre CNN ne fait pas mieux que lui, c'est qu'il y a un problème.

Architecture : **Image → Flatten → Dense(128) + ReLU → Dense(7) + Softmax**

### 2.1 Comment ça marche

**L'image en entrée.** Une couche Dense ne comprend pas les images en 2D : elle veut une simple liste de nombres. La couche `Flatten` "déroule" l'image 48×48 en un vecteur de 48 × 48 = **2 304 valeurs**. On perd au passage l'information de voisinage entre pixels (le réseau ne sait plus quel pixel est à côté de quel autre) : c'est justement ce que le CNN va corriger.

**Poids et biais.** Chaque neurone d'une couche Dense est relié à toutes les entrées. Il calcule une somme pondérée : `z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b`. Les **poids** `w` disent quelle importance donner à chaque entrée, le **biais** `b` décale le résultat. Ce sont eux que le réseau apprend.

**Fonction d'activation.** On applique ensuite une fonction non linéaire à `z`. On utilise **ReLU** : `ReLU(z) = max(0, z)`. Sans activation, empiler des couches ne servirait à rien : une suite d'opérations linéaires reste linéaire, et le réseau ne pourrait apprendre que des frontières toutes droites.

**Propagation avant.** L'image passe couche par couche : Flatten → Dense(128) + ReLU → Dense(7) + Softmax. À la fin on obtient 7 probabilités.

**Sortie multiclasse.** La dernière couche a **7 neurones** (un par émotion) avec **Softmax**, qui transforme les 7 valeurs en probabilités positives dont la somme fait 1. La classe prédite est celle qui a la plus grande probabilité.

**Fonction de perte.** Pour mesurer à quel point le réseau se trompe, on utilise la **categorical cross-entropy** : `loss = -log(p_vraie_classe)`. Si le réseau donne 0.9 à la bonne classe, la loss vaut 0.1 (faible). S'il donne 0.05, elle vaut 3 (forte). Elle punit surtout les erreurs faites avec confiance.

**Rétropropagation.** Une fois la loss calculée, on calcule pour chaque poids son **gradient** : de combien la loss change si on bouge ce poids un tout petit peu. On part de la sortie et on remonte couche par couche vers l'entrée grâce à la règle de dérivation en chaîne : c'est la *backpropagation*.

**Descente de gradient.** On bouge ensuite chaque poids un peu dans le sens qui fait **baisser** la loss : `w ← w - learning_rate × gradient`. On répète ça sur des petits paquets d'images (les *batchs*) pendant plusieurs *époques* (une époque = un passage sur tout le train). On utilise **Adam**, une version améliorée de la descente de gradient qui adapte le pas pour chaque poids.

### 2.2 Construction du modèle

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# On fixe la graine aléatoire pour avoir des résultats reproductibles
keras.utils.set_random_seed(42)

baseline = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

baseline.summary()

Le compte des paramètres :
- `Flatten` : 0 paramètre, il ne fait que réorganiser les pixels.
- `Dense(128)` : 2 304 entrées × 128 neurones + 128 biais = **295 040**.
- `Dense(7)` : 128 × 7 + 7 biais = **903**.

Presque tous les paramètres sont dans la première couche, parce que chaque neurone est connecté à chacun des 2 304 pixels.

### 2.3 Entraînement

On entraîne 20 époques avec Adam, des batchs de 64 images, et on suit la loss et l'accuracy sur la validation à chaque époque.

In [ ]:
baseline.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_baseline = baseline.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_baseline.history["loss"], label="train")
axes[0].plot(history_baseline.history["val_loss"], label="validation")
axes[0].set_title("Loss — modèle dense")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history_baseline.history["accuracy"], label="train")
axes[1].plot(history_baseline.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy — modèle dense")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.show()

print("Accuracy validation finale :", round(history_baseline.history["val_accuracy"][-1], 3))
print("Pour rappel, toujours répondre 'happy' donnerait ~0.25")

**Ce qu'on retient :**
- Le modèle dense apprend un peu : ~37 % en validation, mieux que les ~25 % qu'on aurait en répondant toujours "happy". Mais ça reste faible.
- Il plafonne vite : la validation n'avance plus après ~8 époques, alors que le train continue de s'améliorer. Les deux courbes de loss s'écartent : c'est un début de **surapprentissage** (le modèle commence à apprendre le train par cœur au lieu de généraliser).
- Comme il voit l'image comme une liste de pixels sans structure, il ne peut pas repérer une bouche ou des sourcils où qu'ils soient dans l'image.

C'est notre score à battre avec le CNN.

## Partie 3 — Construction d'un CNN

### 3.1 Les briques d'un CNN

**Filtres et convolution.** Un filtre est une petite grille de poids (ici 3×3). On le fait glisser sur toute l'image : à chaque position, on multiplie les 9 pixels sous le filtre par les 9 poids et on additionne. Si la zone ressemble au motif du filtre (un bord horizontal, un coin...), le résultat est grand. Les poids des filtres ne sont pas choisis à la main : ils sont **appris** pendant l'entraînement, comme les poids d'une couche Dense.

**Feature maps.** Le résultat d'un filtre passé sur toute l'image est une nouvelle "image" qui montre où le motif a été trouvé : c'est une **feature map**. Avec 32 filtres, on obtient 32 feature maps.

**Taille du kernel.** C'est la taille du filtre. On prend **3×3** : assez petit pour peu de paramètres, et en empilant les couches, le réseau finit quand même par voir de grandes zones de l'image.

**Stride.** Le pas avec lequel le filtre se déplace. On garde **stride = 1** (le filtre avance pixel par pixel) et on laisse le pooling réduire la taille.

**Padding.** Sans rien faire, un filtre 3×3 ne peut pas se centrer sur les pixels du bord, et l'image rétrécit de 2 pixels à chaque convolution. Avec `padding="same"`, on ajoute une bordure de zéros autour de l'image pour **garder la même taille** en sortie.

**ReLU.** Comme pour le modèle dense, on applique `max(0, z)` après chaque convolution pour ajouter de la non-linéarité. Ça garde seulement les endroits où le motif est présent (valeurs positives).

**Pooling.** Le **MaxPooling 2×2** découpe chaque feature map en carrés de 2×2 et garde juste le max de chaque carré. La taille est **divisée par 2**. Ça réduit le calcul et rend le réseau un peu insensible aux petits décalages (si la bouche bouge d'un pixel, le max reste à peu près le même).

**Flatten + Dense.** Une fois les caractéristiques extraites, on "déroule" les feature maps en vecteur et on les donne à des couches Dense classiques qui décident de l'émotion.

**Couche de sortie.** Comme pour le modèle dense : **Dense(7) + Softmax** pour avoir une probabilité par émotion.

### 3.2 Notre architecture

On empile **3 blocs Conv + Pool** avec de plus en plus de filtres (32 → 64 → 128) :
- les premières couches repèrent des choses simples (bords, contrastes) ;
- les suivantes combinent ces motifs en formes plus complexes (yeux, coins de la bouche, sourcils froncés).

On double le nombre de filtres à chaque fois que le pooling divise la taille par 2, pour ne pas perdre trop d'information. 3 blocs, c'est un bon compromis pour des images de 48×48 : après 3 poolings, on est à 6×6, en dessous ça deviendrait trop petit.

In [ ]:
keras.utils.set_random_seed(42)

cnn = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    # Bloc 1 : motifs simples (bords, contrastes)
    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 2 : motifs un peu plus complexes
    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 3 : parties du visage
    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Classification
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

cnn.summary()

### 3.3 Évolution des dimensions et nombre de paramètres

Pour une Conv2D, le nombre de paramètres vaut : `(taille kernel × taille kernel × nb canaux en entrée + 1) × nb filtres` (le +1 c'est le biais de chaque filtre).

| Couche | Sortie | Calcul des paramètres | Paramètres |
|---|---|---|---|
| Entrée | 48 × 48 × 1 | — | 0 |
| Conv2D 32 filtres 3×3 | 48 × 48 × 32 | (3×3×1 + 1) × 32 | 320 |
| MaxPooling 2×2 | 24 × 24 × 32 | — | 0 |
| Conv2D 64 filtres 3×3 | 24 × 24 × 64 | (3×3×32 + 1) × 64 | 18 496 |
| MaxPooling 2×2 | 12 × 12 × 64 | — | 0 |
| Conv2D 128 filtres 3×3 | 12 × 12 × 128 | (3×3×64 + 1) × 128 | 73 856 |
| MaxPooling 2×2 | 6 × 6 × 128 | — | 0 |
| Flatten | 4 608 | — | 0 |
| Dense 128 | 128 | 4 608 × 128 + 128 | 589 952 |
| Dense 7 (Softmax) | 7 | 128 × 7 + 7 | 903 |
| **Total** | | | **683 527** |

**Ce qu'on remarque :**
- Grâce au `padding="same"`, la taille ne bouge pas dans les Conv2D, seul le nombre de canaux change. C'est le pooling qui divise la taille par 2.
- Les couches de convolution ont **peu de paramètres** (un filtre 3×3 est réutilisé partout sur l'image), alors que la première Dense en a à elle seule **86 %**. C'est une piste pour plus tard si le modèle sur-apprend.

### 3.4 À quoi ressemblent les feature maps ?

On fait passer un visage dans la première couche de convolution et on affiche les 32 feature maps produites.

Attention : le réseau n'est **pas encore entraîné**, ses filtres sont donc aléatoires. On voit déjà qu'une convolution fait ressortir des zones différentes de l'image (bords, zones claires ou sombres). On refera l'exercice après l'entraînement pour comparer.

In [ ]:
# Un mini-modèle qui s'arrête après la première couche de convolution
extracteur = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)

visage = X_train[0:1]  # on garde la dimension batch : (1, 48, 48, 1)
feature_maps = extracteur.predict(visage, verbose=0)
print("Forme des feature maps :", feature_maps.shape)

plt.imshow(visage[0].squeeze(), cmap="gray")
plt.title(f"Image d'entrée : {classes[y_train[0]]}")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.axis("off")
plt.suptitle("Les 32 feature maps de la 1re couche (filtres aléatoires, avant entraînement)")
plt.show()

**Ce qu'on remarque :**
- Chaque filtre fait ressortir quelque chose de différent : certains mettent en avant les contours (yeux, bouche), d'autres les zones claires ou sombres.
- Beaucoup de feature maps sont presque entièrement noires : c'est l'effet du **ReLU**, qui met à 0 toutes les valeurs négatives. Avec des filtres aléatoires, beaucoup ne "réagissent" à rien sur cette image.
- La taille reste bien 48×48 grâce au `padding="same"`.

## Partie 4 — Entraînement du CNN

### 4.1 Nos choix

| Choix | Valeur | Pourquoi |
|---|---|---|
| **Fonction de perte** | `categorical_crossentropy` | C'est la perte standard en classification multiclasse avec Softmax et des labels one-hot : elle compare la distribution prédite (7 probas) à la vraie (un 1 à la bonne classe). |
| **Optimiseur** | Adam (learning rate 0.001 par défaut) | Il adapte le pas pour chaque poids et converge vite sans trop de réglages. C'est le choix par défaut raisonnable. |
| **Batch size** | 64 | Assez grand pour que le gradient soit stable et le calcul rapide, assez petit pour faire beaucoup de mises à jour par époque (24 402 / 64 ≈ 382 mises à jour). |
| **Époques** | 20 | Assez pour voir le modèle converger **et** voir apparaître un éventuel surapprentissage. |
| **Métrique** | accuracy | Simple à lire pour suivre l'entraînement. Mais vu le déséquilibre des classes, on regardera aussi les scores par classe. |

On garde exactement les mêmes réglages que pour le modèle dense, pour que la comparaison soit juste.

### 4.2 Entraînement

Sur Colab, on pourrait activer le GPU pour accélerer l'apprentissage.

In [ ]:
cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_cnn = cnn.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

### 4.3 Courbes d'apprentissage

On trace le CNN et le modèle dense sur les mêmes graphiques pour comparer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_cnn.history["loss"], label="CNN - train", color="tab:blue")
axes[0].plot(history_cnn.history["val_loss"], label="CNN - validation", color="tab:orange")
axes[0].plot(history_baseline.history["val_loss"], label="Dense - validation", color="gray", linestyle="--")
axes[0].set_title("Loss")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history_cnn.history["accuracy"], label="CNN - train", color="tab:blue")
axes[1].plot(history_cnn.history["val_accuracy"], label="CNN - validation", color="tab:orange")
axes[1].plot(history_baseline.history["val_accuracy"], label="Dense - validation", color="gray", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.show()

meilleure_epoque = int(np.argmin(history_cnn.history["val_loss"]))
print("Meilleure époque (val_loss la plus basse) :", meilleure_epoque + 1)
print("  val_accuracy à cette époque :", round(history_cnn.history["val_accuracy"][meilleure_epoque], 3))
print("val_accuracy à la fin :", round(history_cnn.history["val_accuracy"][-1], 3))
print("train_accuracy à la fin :", round(history_cnn.history["accuracy"][-1], 3))

**Ce qu'on voit :**
- Le CNN fait **beaucoup mieux** que le modèle dense : ~53 % en validation contre ~37 %. Les convolutions repèrent des motifs locaux (bouche, yeux) où qu'ils soient, le dense ne pouvait pas.
- Mais il y a un **gros surapprentissage** :
  - la loss de validation baisse jusqu'à l'**époque 5**, puis remonte en flèche (de 1.25 à plus de 3) ;
  - pendant ce temps, l'accuracy de train monte jusqu'à **93 %** alors que la validation stagne autour de 52 %.
- Concrètement : après l'époque 5, le réseau n'apprend plus à reconnaître des émotions, il **apprend par cœur** les images du train. Avec ~680 000 paramètres pour 24 000 images, il a largement la place de le faire.
- La loss de validation qui explose alors que l'accuracy bouge peu veut dire que le modèle devient **trop sûr de lui** : quand il se trompe, il se trompe avec une proba proche de 1, et la cross-entropy punit très fort ça.

### 4.4 Les feature maps après entraînement

Même visage, même couche qu'en 3.4, mais maintenant les filtres ont été appris.

In [ ]:
# Le mini-modèle "extracteur" partage les poids du CNN : il est donc entraîné lui aussi
feature_maps = extracteur.predict(visage, verbose=0)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.axis("off")
plt.suptitle("Les 32 feature maps de la 1re couche (après entraînement)")
plt.show()

les feature maps sont maintenant beaucoup plus "ciblées". Plusieurs filtres réagissent surtout aux **yeux**, aux **sourcils** et à la **bouche ouverte**, c'est-à-dire précisément les zones qui portent l'expression. D'autres font ressortir les contours du visage. Le réseau a appris tout seul où regarder.

## Partie 5 — Évaluation et analyse des erreurs

On passe enfin sur le **jeu de test**, que les modèles n'ont jamais vu. Les modèles de la partie 6 seront eux comparés sur la validation, et seul le modèle final repassera sur le test.

### 5.1 Score global sur le test

In [ ]:
loss_dense, acc_dense = baseline.evaluate(X_test, y_test_oh, verbose=0)
loss_cnn, acc_cnn = cnn.evaluate(X_test, y_test_oh, verbose=0)

pd.DataFrame({
    "loss test": [loss_dense, loss_cnn],
    "accuracy test": [acc_dense, acc_cnn],
}, index=["Modèle dense", "CNN"]).round(3)

**Ce qu'on voit :**
- Sur le test, le CNN atteint **~52 %** contre **~38 %** pour le modèle dense : +14 points. Les scores sont proches de ceux de la validation, donc notre validation était une bonne estimation.
- Pour situer : sur 7 classes, le hasard ferait ~14 %, et toujours répondre "happy" ~25 %.
- La loss de test du CNN (~3.2) est pire que celle du dense (~1.6) alors que son accuracy est bien meilleure : c'est la trace du surapprentissage vu plus haut, le CNN donne des probas extrêmes même quand il a tort.

### 5.2 Matrice de confusion

L'accuracy globale cache beaucoup de choses. La **matrice de confusion** montre, pour chaque vraie classe (ligne), dans quelle classe le modèle l'a rangée (colonne). La diagonale = les bonnes réponses.

On la normalise par ligne : chaque case donne le **pourcentage** des images de la vraie classe. Sinon `happy`, qui a beaucoup plus d'images, écraserait tout le reste.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

probas_test = cnn.predict(X_test, verbose=0)   # (7178, 7) : les 7 probas pour chaque image
y_pred = np.argmax(probas_test, axis=1)        # la classe avec la plus grande proba

cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Matrice de confusion du CNN (test, normalisée par ligne)")
plt.show()

**Ce qu'on lit dans la matrice :**
- **happy** (75 %) et **surprise** (66 %) sont de loin les mieux reconnues. Ce sont des expressions très marquées (grand sourire, bouche ouverte + yeux écarquillés), et `happy` a en plus le plus d'exemples.
- **sad**, **fear**, **angry** et **neutral** tournent autour de 40 % : le modèle se trompe plus souvent qu'il n'a raison sur ces classes.
- Les grosses confusions sont entre **sad / fear / neutral** (cases à ~0.2 hors diagonale) : ce sont des expressions plus subtiles, qui se jouent sur de petits détails (coin de la bouche, sourcils).
- `disgust` est souvent pris pour **angry** (23 %) : les deux froncent les sourcils et plissent le visage.
- `surprise` est parfois prise pour **fear** (14 %) : les deux ont les yeux grands ouverts et la bouche ouverte.

### 5.3 Scores par classe

- **Precision** : quand le modèle dit "happy", à quel point il a raison.
- **Recall** : parmi toutes les vraies images "happy", combien il en retrouve.
- **F1-score** : un mélange des deux (moyenne harmonique).
- **Support** : le nombre d'images de la classe dans le test.

In [ ]:
print(classification_report(y_test, y_pred, target_names=classes, digits=2))

**Ce qu'on retient :**
- Le classement par F1-score confirme la matrice : **happy** (0.74) et **surprise** (0.68) loin devant, **fear** (0.37) et **sad** (0.39) les plus difficiles.
- Surprise : **disgust** s'en sort à 0.45 malgré seulement 436 images d'entraînement. Mais avec 111 images de test, ce score est fragile : quelques images en plus ou en moins le feraient bouger de plusieurs points.
- `neutral` a une precision (0.49) plus haute que son recall (0.41) : le modèle hésite à dire "neutral", et range beaucoup de visages neutres ailleurs (sad, fear).
- La **macro avg** (0.50) donne le même poids à chaque classe, alors que la **weighted avg** (0.52) est tirée vers le haut par `happy`. Elles sont proches ici, donc le déséquilibre ne fausse pas trop le score global.

### 5.4 Exemples de prédictions

Sous chaque image : la vraie classe, la classe prédite et la probabilité donnée à cette prédiction. En vert = bonne réponse, en rouge = erreur.

In [ ]:
np.random.seed(3)
indices = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, idx in zip(axes.flat, indices):
    vrai = classes[y_test[idx]]
    pred = classes[y_pred[idx]]
    proba = probas_test[idx, y_pred[idx]]
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"vrai : {vrai}\npréd : {pred} ({proba:.2f})",
                 color="green" if vrai == pred else "red", fontsize=10)
    ax.axis("off")
plt.suptitle("Prédictions au hasard sur le test")
plt.show()

On retrouve ce qu'on a vu dans la matrice : les sourires et les bouches grandes ouvertes sont bien reconnus. Le modèle met souvent des probas à **1.00**, même quand il a tort (ex. l'enfant "angry" qui crie, prédit "happy" à 1.00) : c'est encore un signe du surapprentissage.

### 5.5 Analyse des erreurs

On regarde maintenant uniquement les erreurs, en commençant par celles où le modèle était **le plus sûr de lui** : ce sont souvent les plus instructives.

In [ ]:
erreurs = np.where(y_pred != y_test)[0]
print("Nombre d'erreurs :", len(erreurs), "sur", len(y_test))

# On trie les erreurs par probabilité décroissante
confiance = probas_test[erreurs, y_pred[erreurs]]
pires = erreurs[np.argsort(-confiance)][:12]

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, idx in zip(axes.flat, pires):
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"vrai : {classes[y_test[idx]]}\npréd : {classes[y_pred[idx]]} ({probas_test[idx, y_pred[idx]]:.2f})",
                 color="red", fontsize=10)
    ax.axis("off")
plt.suptitle("Les 12 erreurs les plus confiantes")
plt.show()

Et les confusions les plus fréquentes (en nombre d'images) :

In [ ]:
cm_brute = confusion_matrix(y_test, y_pred)
np.fill_diagonal(cm_brute, 0)   # on enlève les bonnes réponses

confusions = []
for i in range(len(classes)):
    for j in range(len(classes)):
        confusions.append((classes[i], classes[j], cm_brute[i, j]))

pd.DataFrame(confusions, columns=["vraie classe", "prédite", "nb d'images"]) \
    .sort_values("nb d'images", ascending=False).head(8).reset_index(drop=True)

### 5.6 Bilan de l'analyse

**Les confusions les plus fréquentes** sont `sad → fear`, `neutral → sad`, `fear → sad` et `sad → neutral`. Le trio **sad / fear / neutral** est le vrai point faible du modèle.

**Pourquoi ces classes sont difficiles :**
1. **Les expressions se ressemblent** : un visage triste et un visage neutre ne diffèrent parfois que par le coin de la bouche. En 48×48 pixels, c'est quelques pixels.
2. **Les labels de FER-2013 sont discutables** : en regardant les erreurs "les plus confiantes", beaucoup d'images notées `surprise` ou `neutral` montrent clairement un **grand sourire**. Le modèle dit "happy" à 1.00, et honnêtement il n'a pas tort. Une partie des erreurs vient donc du dataset, pas du modèle.
3. **Certaines images sont spéciales** : dessins, maquillage, visages de profil, filigranes.
4. **Le surapprentissage** : le modèle a mémorisé le train au lieu de bien généraliser, et il est beaucoup trop confiant.

## Partie 6 — Expérimentations et amélioration

Le gros problème de notre CNN, c'est le **surapprentissage** : il apprend le train par cœur dès l'époque 5. On va faire des expériences pour essayer de l'améliorer.

Règle : on ne change **qu'une seule chose à la fois** par rapport à l'expérience précédente. Comme ça, si le score bouge, on sait d'où ça vient.

| Expérience | Ce qui change |
|---|---|
| 0 (référence) | le CNN de la partie 3 (kernel 3×3, 20 époques) |
| 1 | + **Dropout** |
| 2 | **taille du kernel** 3×3 → 5×5 |
| 3 | **nombre d'époques** réduit |

Toutes les expériences sont comparées sur la **validation**. Le test ne sert qu'à la toute fin, pour le modèle final.

Petite fonction pour tracer les courbes, pour ne pas recopier le même code à chaque expérience :

In [ ]:
def tracer_courbes(history, titre):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title("Loss — " + titre)
    axes[0].set_xlabel("Époque")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title("Accuracy — " + titre)
    axes[1].set_xlabel("Époque")
    axes[1].legend()
    plt.show()

### 6.1 Expérience 1 : Dropout

Pendant l'entraînement, le Dropout "éteint" au hasard une partie des neurones à chaque batch (ils sortent 0). Le réseau ne peut donc plus compter sur un neurone précis pour mémoriser une image : il est obligé de répartir l'information et d'apprendre des motifs plus généraux. Au moment de prédire, tous les neurones sont rallumés.

On en met :
- **0.25** (25 % des neurones éteints) après chaque bloc de convolution ;
- **0.5** avant la couche de sortie, juste après la grosse couche Dense qui contient 86 % des paramètres : c'est là que le modèle mémorise le plus.

Tout le reste est identique au CNN de la partie 3.

In [ ]:
keras.utils.set_random_seed(42)

cnn_dropout = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax"),
])

cnn_dropout.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history_dropout = cnn_dropout.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_dropout, "Exp. 1 : Dropout")

**Ce qu'on voit :**
- Le surapprentissage a quasiment **disparu** : les courbes de train et de validation restent collées, la loss de validation descend jusqu'au bout (1.14 à l'époque 18) au lieu de remonter jusqu'à 3.
- L'accuracy de validation monte à **~57 %**, contre ~53 % au mieux pour le CNN de base.
- L'accuracy de train est beaucoup plus basse qu'avant (~59 % contre 93 %) : c'est normal, le Dropout rend l'entraînement plus difficile exprès. Ce qui compte, c'est la validation.
- Les courbes montent encore à l'époque 20 : le modèle n'a pas fini d'apprendre. Avec le Dropout, il apprend plus lentement mais plus "proprement".

### 6.2 Expérience 2 : kernel 5×5

**L'idée :** un filtre 3×3 ne voit que 9 pixels à la fois. Un filtre **5×5** en voit 25 : chaque convolution capte une zone plus large du visage d'un coup (un coin de bouche entier, un sourcil entier). En contrepartie, chaque filtre a presque 3 fois plus de poids, donc le modèle est plus lourd et plus lent.

On garde le modèle de l'expérience 1 (avec Dropout) et on change **uniquement** `kernel_size=3` → `kernel_size=5` dans les 3 couches de convolution. Grâce au `padding="same"`, les dimensions de sortie ne changent pas.

Nouveau nombre de paramètres des convolutions :

| Couche | Kernel 3×3 | Kernel 5×5 |
|---|---|---|
| Conv2D 32 | (3×3×1 + 1) × 32 = 320 | (5×5×1 + 1) × 32 = 832 |
| Conv2D 64 | (3×3×32 + 1) × 64 = 18 496 | (5×5×32 + 1) × 64 = 51 264 |
| Conv2D 128 | (3×3×64 + 1) × 128 = 73 856 | (5×5×64 + 1) × 128 = 204 928 |
| **Total du modèle** | **683 527** | **847 879** |

In [ ]:
keras.utils.set_random_seed(42)

cnn_k5 = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(128, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax"),
])

cnn_k5.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
print("Nombre de paramètres :", cnn_k5.count_params())

history_k5 = cnn_k5.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_k5, "Exp. 2 : kernel 5×5")

**Ce qu'on voit :**
- En accuracy, le kernel 5×5 fait **jeu égal** avec le 3×3 : ~57 % de validation à la fin pour les deux (0.569 contre 0.564, soit ~20 images d'écart sur 4 307 : c'est du bruit, l'accuracy de validation bouge d'autant d'une époque à l'autre).
- En revanche sa **loss de validation est moins bonne** (1.21 contre 1.16) et il recommence à **sur-apprendre** à partir de l'époque ~13 : la loss de validation remonte doucement pendant que celle du train continue de baisser. Avec 24 % de paramètres en plus, il a plus de "place" pour mémoriser.
- Et il est **2 fois plus lent** à entraîner (~25-35 s par époque contre ~13 s).

Sur des images de 48×48, les détails importants (coins de la bouche, sourcils) sont petits : des filtres 3×3 empilés suffisent, et deux 3×3 à la suite voient de toute façon une zone de 5×5. Plus gros ne veut pas dire meilleur ici.

### 6.3 Expérience 3 : moins d'époques

**L'idée :** au lieu d'ajouter du Dropout, une autre façon d'éviter le surapprentissage est simplement de **s'arrêter plus tôt**, avant que le modèle commence à apprendre par cœur.

Sur le Dropout ça n'aurait pas de sens (il apprenait encore à l'époque 20, on le couperait en plein apprentissage). On l'applique donc au **CNN de base** (partie 3, sans Dropout), qui lui sur-apprenait dès l'époque 5 : on l'entraîne **5 époques au lieu de 20**, c'est-à-dire pile là où sa loss de validation était au plus bas.

Seul le nombre d'époques change par rapport à l'expérience 0. On pourra comparer les deux remèdes au surapprentissage : s'arrêter tôt (exp. 3) contre le Dropout (exp. 1).

In [ ]:
keras.utils.set_random_seed(42)

cnn_court = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

cnn_court.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history_court = cnn_court.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=5,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_court, "Exp. 3 : CNN de base, 5 époques")

**Ce qu'on voit :** les 5 époques reproduisent exactement le début de l'expérience 0 (même graine aléatoire) : on s'arrête à **53.4 %** de validation, avec une val_loss de 1.25, juste avant que le surapprentissage commence. Les courbes de train et de validation restent proches : le surapprentissage est évité, mais le modèle n'a pas eu le temps d'aller plus loin.

### 6.4 Tableau comparatif

Pour chaque expérience, on regarde l'état du modèle **à la fin de l'entraînement** (c'est ce modèle-là qu'on garde, puisqu'on n'utilise pas d'Early Stopping) :
- la loss et l'accuracy de validation ;
- l'**écart** entre accuracy de train et de validation : plus il est grand, plus le modèle sur-apprend.

In [ ]:
experiences = {
    "0 - CNN de base (20 ép.)": history_cnn,
    "1 - + Dropout (20 ép.)": history_dropout,
    "2 - + kernel 5×5 (20 ép.)": history_k5,
    "3 - CNN de base (5 ép.)": history_court,
}

lignes = []
for nom, h in experiences.items():
    lignes.append({
        "expérience": nom,
        "val_loss": h.history["val_loss"][-1],
        "val_accuracy": h.history["val_accuracy"][-1],
        "train_accuracy": h.history["accuracy"][-1],
        "écart train - val": h.history["accuracy"][-1] - h.history["val_accuracy"][-1],
    })

tableau = pd.DataFrame(lignes).set_index("expérience").round(3)
tableau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for nom, h in experiences.items():
    axes[0].plot(h.history["val_loss"], label=nom)
    axes[1].plot(h.history["val_accuracy"], label=nom)
axes[0].set_title("Loss de validation")
axes[1].set_title("Accuracy de validation")
for ax in axes:
    ax.set_xlabel("Époque")
    ax.legend()
plt.show()

### 6.5 Choix du modèle final

| Expérience | Modification | Observation |
|---|---|---|
| 0 | CNN de base, 20 époques | Gros surapprentissage : 93 % en train mais ~52 % en validation, et la val_loss explose (>3). Le modèle est beaucoup trop confiant. |
| 1 | + Dropout | Surapprentissage quasi éliminé (train et validation à quelques points d'écart), **meilleure val_loss** (~1.16), ~56-57 % en validation. Apprend encore à l'époque 20. |
| 2 | kernel 5×5 | Même accuracy que l'exp. 1 (à 0.5 point près), mais val_loss moins bonne, recommence à sur-apprendre vers l'époque 13, 24 % de paramètres en plus et 2× plus lent. |
| 3 | CNN de base, 5 époques | Évite le surapprentissage, mais plafonne plus bas (~53 %) : on arrête le modèle avant qu'il ait eu le temps d'apprendre des choses utiles. |

**Modèle retenu : l'expérience 1 (CNN 3×3 + Dropout, 20 époques).**
- Les expériences 1 et 2 sont à égalité en accuracy : l'écart est trop petit pour être significatif. Pour les départager, on regarde le reste.
- L'exp. 1 a la **meilleure loss de validation** : ses probabilités sont plus fiables (il se trompe moins "avec confiance").
- Ses courbes de train et de validation restent proches : il **généralise** au lieu de mémoriser, alors que le 5×5 commence déjà à décrocher.
- À performance égale, on prend le modèle le plus **simple** : moins de paramètres et 2× plus rapide. Ça comptera aussi pour la vidéo (partie 9), où il faut prédire vite.
- S'arrêter tôt (exp. 3) limite aussi le surapprentissage, mais c'est moins efficace : le Dropout permet d'entraîner plus longtemps sans sur-apprendre, et donc d'aller plus loin.
- Piste pour aller plus loin : comme ses courbes montaient encore à l'époque 20, quelques époques de plus pourraient encore l'améliorer un peu.

### 6.6 Le modèle final sur le test

On évalue le modèle retenu sur le test (une seule fois !) et on le compare au modèle dense et au CNN de base de la partie 5.

In [42]:
cnn_final = cnn_dropout

loss_final, acc_final = cnn_final.evaluate(X_test, y_test_oh, verbose=0)

pd.DataFrame({
    "loss test": [loss_dense, loss_cnn, loss_final],
    "accuracy test": [acc_dense, acc_cnn, acc_final],
}, index=["Modèle dense", "CNN de base", "CNN final (Dropout)"]).round(3)

NameError: name 'cnn_dropout' is not defined

In [ ]:
from sklearn.metrics import f1_score

y_pred_final = np.argmax(cnn_final.predict(X_test, verbose=0), axis=1)

# F1-score par classe, avant / après
f1_avant = f1_score(y_test, y_pred, average=None)
f1_apres = f1_score(y_test, y_pred_final, average=None)

x = np.arange(len(classes))
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, f1_avant, width=0.4, label="CNN de base")
plt.bar(x + 0.2, f1_apres, width=0.4, label="CNN final")
plt.xticks(x, classes)
plt.ylabel("F1-score")
plt.title("F1-score par classe sur le test")
plt.legend()
plt.show()

cm_final = confusion_matrix(y_test, y_pred_final, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm_final, display_labels=classes).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Matrice de confusion du CNN final (test)")
plt.show()

**Ce qu'on voit :**
- Le CNN final gagne **+5.5 points** sur le test (**57.6 %** contre 52.1 %), et sa loss est presque divisée par 3 (1.13 contre 3.19) : ses probabilités sont beaucoup plus fiables, il ne se trompe plus "avec une confiance de 1.00".
- Le score de test (57.6 %) est proche de celui de validation (~56-57 %) : le choix fait sur la validation se confirme sur des images jamais vues.
- Le F1-score progresse pour **angry, happy, neutral, sad et surprise**. `neutral` (+0.09) et `sad` (+0.08) gagnent le plus.
- Mais il **baisse pour fear et disgust** :
  - `disgust` (111 images de test) est maintenant souvent prédit `angry` (31 %) : avec seulement 436 images d'entraînement, le Dropout rend cette petite classe encore plus difficile à apprendre, et le modèle se rabat sur la classe voisine plus fréquente ;
  - `fear` reste la classe la plus difficile : elle est confondue avec `sad` (28 %), et même avec `surprise` (12 %).
- Le trio `sad` / `neutral` / `fear` reste la principale source de confusions, comme en partie 5.

Le modèle final n'est donc pas meilleur partout, mais il est meilleur **globalement** et surtout beaucoup plus **fiable** dans ses probabilités. C'est lui qu'on utilise pour la suite.

On sauvegarde le modèle final pour pouvoir le réutiliser plus tard (détection de visages, vidéo) sans le ré-entraîner :

In [ ]:
cnn_final.save("modele_final.keras")
print("Modèle sauvegardé !")

## Partie 7 — Enrichissement : la détection d'objets avec YOLO

Jusqu'ici, notre CNN répond à la question **"quelle expression ?"** sur un visage **déjà découpé**. Dans une vraie photo, il faut d'abord répondre à **"où sont les visages ?"**. C'est le rôle de la **détection d'objets**, et on va utiliser **YOLO** pour ça.

### 7.1 Classification vs détection

| | Classification | Détection |
|---|---|---|
| Question | quelle classe pour toute l'image ? | où sont les objets, et lesquels ? |
| Sortie | une classe + une proba | pour **chaque** objet : une boîte + une classe + un score |
| Exemple | notre CNN : "happy, 0.92" | YOLO : "visage en (120, 40, 210, 150), 0.89" |

### 7.2 Les notions clés

- **Bounding box** : le rectangle qui entoure un objet, donné par ses coins `(x1, y1, x2, y2)` en pixels.
- **Confidence score** : à quel point le modèle est sûr qu'il y a bien un objet dans cette boîte (entre 0 et 1). On ne garde que les boîtes au-dessus d'un **seuil** (0.25 par défaut).
- **IoU (Intersection over Union)** : mesure à quel point deux boîtes se superposent. `IoU = aire de l'intersection / aire de l'union`. 1 = boîtes identiques, 0 = aucun recouvrement.
- **NMS (Non-Maximum Suppression)** : le réseau propose souvent **plusieurs boîtes** pour le même objet. La NMS garde la boîte avec le meilleur score et supprime les autres boîtes qui la recouvrent trop (IoU au-dessus d'un seuil, 0.7 par défaut).
- **Precision / recall / mAP** : la *precision* dit quelle part des boîtes détectées sont de vrais visages, le *recall* quelle part des vrais visages ont été trouvés. La **mAP** résume les deux sur tous les seuils de confiance : c'est la métrique standard pour comparer des détecteurs. Pour la calculer, il faut des images annotées à la main (avec les vraies boîtes) : on ne la calcule pas ici, on se fie à celle du modèle pré-entraîné.

### 7.3 Le principe de YOLO

**YOLO = You Only Look Once.** Les anciens détecteurs regardaient l'image des centaines de fois (une fois par zone candidate). YOLO, lui, fait passer l'image **une seule fois** dans un CNN :
1. l'image est découpée en une **grille** ;
2. pour chaque case, le réseau prédit directement des boîtes, leur score de confiance et leur classe ;
3. on filtre les boîtes trop peu confiantes, puis la NMS enlève les doublons.

Tout se fait en un seul passage, d'où sa rapidité : c'est ce qui permet de l'utiliser sur de la vidéo (partie 9).

### 7.4 Quel modèle ?

Un YOLO "standard" est pré-entraîné sur le dataset **COCO** (80 classes : personne, voiture, chien...). Il détecte des **personnes**, mais **pas des visages**. On utilise donc un modèle **YOLOv8 déjà entraîné à détecter des visages**, partagé sur Hugging Face ([arnabdhar/YOLOv8-Face-Detection](https://huggingface.co/arnabdhar/YOLOv8-Face-Detection)). C'est du **Transfer Learning** côté détection : quelqu'un est parti d'un YOLOv8 pré-entraîné et l'a fine-tuné sur des visages.

C'est la version **nano** (6 Mo) : elle tourne sans problème sur CPU. Pas besoin de l'entraîner, on l'utilise telle quelle.

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS   # 2 photos d'exemple fournies avec ultralytics
from huggingface_hub import hf_hub_download
import matplotlib.patches as patches

chemin_yolo = hf_hub_download("arnabdhar/YOLOv8-Face-Detection", "model.pt")
yolo = YOLO(chemin_yolo)

print("Classes que ce YOLO sait détecter :", yolo.names)

### 7.5 Première détection

On teste sur une photo d'exemple fournie avec la librairie. Petite fonction pour dessiner des boîtes, qu'on va réutiliser plusieurs fois :

In [ ]:
def dessiner_boites(image, boites, textes, titre=""):
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    for (x1, y1, x2, y2), texte in zip(boites, textes):
        plt.gca().add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime", linewidth=2))
        plt.text(x1, y1 - 5, texte, color="black", backgroundcolor="lime", fontsize=9)
    plt.title(titre)
    plt.axis("off")
    plt.show()


photo = Image.open(ASSETS / "zidane.jpg").convert("RGB")
resultat = yolo(photo, verbose=False)[0]

boites = resultat.boxes.xyxy.numpy()   # une ligne par visage : x1, y1, x2, y2
scores = resultat.boxes.conf.numpy()   # le score de confiance de chaque boîte

print("Nombre de visages détectés :", len(boites))
for b, s in zip(boites, scores):
    print("  boîte :", b.round(), "| confiance :", round(float(s), 2))

dessiner_boites(photo, boites, [f"visage {s:.2f}" for s in scores], "Détection de visages avec YOLO")

YOLO trouve les **2 visages**. Celui de gauche a un score bien plus faible (~0.37) : il est vu **de profil**, avec la bouche grande ouverte. Ça ressemble moins aux visages de face qu'il a vus à l'entraînement, il est donc moins sûr de lui.

### 7.6 La NMS en action

Pour voir à quoi sert la NMS, on la "désactive" en mettant son seuil d'IoU à 1.0 (deux boîtes ne seront jamais considérées comme des doublons) et on baisse le seuil de confiance à 0.05 :

In [ ]:
sans_nms = yolo(photo, conf=0.05, iou=1.0, verbose=False)[0]
avec_nms = yolo(photo, conf=0.05, iou=0.7, verbose=False)[0]

print("Boîtes sans NMS :", len(sans_nms.boxes))
print("Boîtes avec NMS :", len(avec_nms.boxes))

dessiner_boites(photo, sans_nms.boxes.xyxy.numpy(), ["" for _ in sans_nms.boxes], "Sans NMS : plein de boîtes en double")

Sans NMS, le réseau propose une vingtaine de boîtes, presque toutes empilées sur les 2 mêmes visages. La NMS ne garde que la meilleure de chaque groupe.

### 7.7 Calcul de l'IoU à la main

L'IoU est au cœur de la NMS (et de l'évaluation des détecteurs). On la calcule nous-mêmes entre la meilleure boîte et une autre boîte proposée pour le même visage :

In [ ]:
def iou(a, b):
    # coins du rectangle d'intersection
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (a[2] - a[0]) * (a[3] - a[1])
    aire_b = (b[2] - b[0]) * (b[3] - b[1])
    return intersection / (aire_a + aire_b - intersection)


boites_sans_nms = sans_nms.boxes.xyxy.numpy()
meilleure = boites_sans_nms[0]   # les boîtes sont triées par score décroissant

ious = [round(float(iou(meilleure, b)), 2) for b in boites_sans_nms[1:]]
print("IoU de chaque boîte avec la meilleure :", ious)

Deux groupes bien nets :
- les boîtes posées sur le **même visage** ont une IoU proche de 1 avec la meilleure : ce sont des doublons, la NMS les supprime ;
- les boîtes sur **l'autre visage** ont une IoU de 0 : elles ne sont pas en concurrence avec la meilleure, la NMS les garde (et garde la meilleure d'entre elles).

### 7.8 Le seuil de confiance : precision vs recall

Le seuil de confiance est un compromis :
- **seuil bas** : on rate moins de visages (meilleur **recall**), mais on risque des fausses détections
- **seuil haut** : les détections sont plus sûres (meilleure **precision**), mais on rate les visages difficiles.

In [ ]:
for seuil in [0.25, 0.5]:
    r = yolo(photo, conf=seuil, verbose=False)[0]
    dessiner_boites(photo, r.boxes.xyxy.numpy(), [f"visage {s:.2f}" for s in r.boxes.conf.numpy()],
                    f"Seuil de confiance = {seuil} → {len(r.boxes)} visage(s)")

Avec un seuil à 0.5, le visage de profil (0.37) disparaît : on a perdu en recall. On garde le seuil par défaut **0.25** pour la suite, pour ne pas rater de visages.

## Partie 8 — Détection et analyse de plusieurs visages

On assemble maintenant tout le système :

**Image → YOLO → boîtes → découpe de chaque visage → prétraitement → notre CNN → expression + proba**

### 8.1 Le point clé : le même prétraitement

Notre CNN a été entraîné sur des images **48×48, en niveaux de gris, avec des pixels entre 0 et 1**. Chaque visage découpé par YOLO doit subir **exactement** la même transformation, sinon le CNN reçoit quelque chose qu'il n'a jamais vu et ses prédictions n'ont aucun sens :
1. découper la boîte dans l'image ;
2. convertir en niveaux de gris ;
3. redimensionner en 48×48 ;
4. diviser par 255 ;
5. mettre en forme `(1, 48, 48, 1)`.

On règle aussi le seuil de la **NMS** à **0.5** au lieu de 0.7 : en testant sur une vidéo, YOLO laissait parfois une petite boîte à l'intérieur d'une grande sur le même visage. Leur IoU (~0.6) était sous le seuil de 0.7, donc la NMS ne les considérait pas comme des doublons. Avec 0.5, plus aucun doublon. Le risque inverse (supprimer un vrai 2e visage) est faible : deux visages se recouvrent rarement à plus de 50 %.

On recharge le modèle final sauvegardé en partie 6 (pas besoin de ré-entraîner) :

In [ ]:
modele = keras.models.load_model("modele_final.keras")


def analyser_image(image):
    """Détecte les visages d'une image RGB et prédit l'expression de chacun."""
    # NMS plus stricte que le défaut (0.5 au lieu de 0.7) : sinon YOLO laisse parfois une petite
    # boîte à l'intérieur d'une grande sur le même visage (vu en vidéo, partie 9)
    resultat = yolo(image, iou=0.5, verbose=False)[0]
    boites = resultat.boxes.xyxy.numpy().astype(int)

    visages = []
    for x1, y1, x2, y2 in boites:
        visage = image.crop((x1, y1, x2, y2))        # 1. découpe
        visage = visage.convert("L")                 # 2. niveaux de gris
        visage = visage.resize((48, 48))             # 3. 48x48
        visage = np.array(visage) / 255.0            # 4. pixels entre 0 et 1
        visages.append(visage.reshape(48, 48, 1))    # 5. bonne forme

    if len(visages) == 0:
        return boites, np.array(visages), np.zeros((0, len(classes)))

    visages = np.array(visages)
    # On appelle le modèle directement plutôt que .predict() : même résultat, mais
    # .predict() a un coût fixe (~150 ms) qui ralentirait beaucoup la webcam en partie 9
    probas = modele(visages, training=False).numpy()
    return boites, visages, probas

### 8.2 Résultat sur une photo

In [ ]:
boites, visages, probas = analyser_image(photo)

textes = [f"{classes[np.argmax(p)]} {p.max():.2f}" for p in probas]
dessiner_boites(photo, boites, textes, "Détection + expression")

### 8.3 Ce que voit vraiment le CNN

Pour chaque visage : la découpe telle que le CNN la reçoit (48×48, gris) et les 7 probabilités du Softmax. Ça permet de vérifier que le prétraitement ressemble bien aux images de FER-2013.

In [ ]:
fig, axes = plt.subplots(len(visages), 2, figsize=(10, 3 * len(visages)), squeeze=False)

for i in range(len(visages)):
    axes[i, 0].imshow(visages[i].squeeze(), cmap="gray")
    axes[i, 0].set_title(f"Visage {i + 1} (48×48)")
    axes[i, 0].axis("off")

    axes[i, 1].barh(classes, probas[i])
    axes[i, 1].set_xlim(0, 1)
    axes[i, 1].set_title(f"Prédiction : {classes[np.argmax(probas[i])]}")

plt.tight_layout()
plt.show()

**Ce qu'on voit :**
- Le visage de gauche (qui crie, bouche grande ouverte, sourcils froncés) est prédit **angry** à 0.56 : cohérent avec l'image.
- Le visage de droite a une expression plus neutre / fermée : le CNN **hésite** entre sad, angry, fear et neutral (~0.2 chacun).
- Les découpes 48×48 ressemblent bien aux images de FER-2013 (visage en gros plan, gris), mais pas parfaitement : le cadrage de YOLO est un peu différent, et le visage de gauche est **de profil**, alors que FER-2013 contient surtout des visages de face.

### 8.4 Sur vos propres photos

Pour tester sur une autre photo (une photo de groupe par exemple) : sur Colab, glissez le fichier dans le panneau "Fichiers" à gauche, puis changez le chemin ci-dessous. Par défaut, on utilise la 2e photo d'exemple d'ultralytics.

In [ ]:
chemin_photo = ASSETS / "bus.jpg"   # remplacer par "ma_photo.jpg" par exemple

photo2 = Image.open(chemin_photo).convert("RGB")
boites, visages, probas = analyser_image(photo2)

print("Visages détectés :", len(boites))
textes = [f"{classes[np.argmax(p)]} {p.max():.2f}" for p in probas]
dessiner_boites(photo2, boites, textes, "Détection + expression")

## Partie 9 — Reconnaissance en direct (webcam)

Une vidéo, c'est une suite d'images. On applique la chaîne de la partie 8 aux images de la **webcam**, en direct :

**Webcam → images successives → YOLO → découpe des visages → CNN → prédictions → affichage**

On utilise **OpenCV** (`cv2`) pour lire la webcam et afficher le résultat dans une fenêtre.

### 9.1 Le problème de la vitesse

La webcam fournit ~30 images par seconde, mais analyser une image coûte cher sur CPU:

| Étape | Temps par image |
|---|---|
| YOLO | ~40 ms (jusqu'à ~200 ms si le CPU est occupé par autre chose) |
| CNN (appel direct, tous les visages d'un coup) | ~20 ms |

Si on analysait chaque image, on plafonnerait vers ~15 images par seconde (et beaucoup moins si le PC est chargé) : ça saccade.

**Solution simple :** on affiche **toutes** les images de la webcam, mais on ne lance l'analyse que sur **une image sur N**. Entre deux analyses, on redessine les dernières boîtes et émotions connues.
- La vidéo reste beaucoup plus fluide.
- Les émotions sont mises à jour plusieurs fois par seconde, largement assez pour une expression qui dure plus longtemps que ça.
- Contrepartie : si quelqu'un bouge vite, la boîte "traîne" un peu derrière le visage.


### 9.2 La boucle en direct

**À lancer en local** (VS Code ou Jupyter sur votre PC) : sur Colab, le code tourne sur un serveur qui n'a ni webcam ni écran, donc la cellule s'arrête avec un message.

Dans la fenêtre : **`s`** pour enregistrer une capture, **`q`** pour quitter.

On peut aussi mettre le chemin d'une vidéo à la place de `0` dans `source` : `cv2.VideoCapture` lit les deux de la même façon.

In [1]:
import sys
import time
import cv2

source = 0               # 0 = webcam, ou "ma_video.mp4"
analyser_tous_les = 3    # on analyse 1 image sur 3

if "google.colab" in sys.modules:
    print("Pas de webcam ni de fenêtre sur Colab : lancez cette partie en local.")
else:
    lecteur = cv2.VideoCapture(source)
    numero = 0
    boites, probas = [], []
    fps = 0
    debut = time.time()

    # try/finally : même si on interrompt la cellule (bouton stop), la webcam est libérée.
    # Sinon elle reste bloquée par le kernel jusqu'à son redémarrage.
    try:
        while True:
            ok, frame = lecteur.read()
            if not ok:   # webcam indisponible ou fin de la vidéo
                break

            # Analyse seulement une image sur N, sinon on garde les derniers résultats
            if numero % analyser_tous_les == 0:
                # OpenCV donne les images en BGR, notre chaîne attend du RGB
                image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                boites, visages, probas = analyser_image(image)

            for (x1, y1, x2, y2), p in zip(boites, probas):
                texte = f"{classes[np.argmax(p)]} {p.max():.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, texte, (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            numero += 1
            fps = numero / (time.time() - debut)
            cv2.putText(frame, f"{fps:.1f} images/s", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            cv2.imshow("Expressions faciales - s : capture, q : quitter", frame)
            touche = cv2.waitKey(1) & 0xFF
            if touche == ord("s"):
                cv2.imwrite("capture_webcam.png", frame)
                print("Capture enregistrée dans capture_webcam.png")
            if touche == ord("q"):
                break

    finally:
        lecteur.release()
        cv2.destroyAllWindows()
    if numero == 0:
        print("Impossible de lire la webcam : elle est peut-être utilisée par une autre appli (Teams, Zoom...)")
        print("ou bloquée dans les paramètres de confidentialité de Windows.")
    else:
        print(f"{numero} images affichées, {fps:.1f} images/s en moyenne")

NameError: name 'Image' is not defined

### 9.3 Une capture

La dernière capture enregistrée avec `s` :

In [1]:
if os.path.exists("capture_webcam.png"):
    plt.figure(figsize=(9, 6))
    plt.imshow(Image.open("capture_webcam.png").convert("RGB"))
    plt.axis("off")
    plt.show()
else:
    print("Pas encore de capture : appuyez sur 's' pendant la démo.")

NameError: name 'os' is not defined

### 9.4 Limites

- **Vitesse** : sur CPU, l'image analysée fait quand même une petite pause dans l'affichage. Pour supprimer ça complètement, il faudrait faire l'analyse dans un autre *thread*, en parallèle de l'affichage, ou utiliser un GPU.
- **Clignotement** : chaque analyse est indépendante des précédentes. Si le CNN hésite entre deux émotions, l'étiquette peut changer d'une analyse à l'autre. On pourrait faire la moyenne des probas sur les dernières analyses, mais il faudrait alors suivre chaque visage d'une image à l'autre (*tracking*).
- **Conditions réelles** : notre CNN a appris sur FER-2013 (visages de face, bien cadrés). Un mauvais éclairage, un visage de profil ou trop loin de la caméra donnent des prédictions moins fiables. Et comme on l'a vu en partie 5, les expressions marquées (joie, surprise) sont bien mieux reconnues que les subtiles (tristesse, peur).

## Conclusion

### Ce qu'on a construit

Un système complet de reconnaissance d'expressions faciales, construit étape par étape :

**Image ou webcam → YOLO (où sont les visages ?) → découpe + prétraitement → notre CNN (quelle expression ?) → affichage**

### Les résultats sur le test

| Modèle | Accuracy test | Loss test |
|---|---|---|
| Toujours répondre "happy" | ~0.25 | — |
| Réseau dense (partie 2) | 0.380 | 1.61 |
| CNN de base (partie 3) | 0.521 | 3.19 |
| **CNN final avec Dropout (partie 6)** | **0.576** | **1.13** |

- Passer du réseau dense au **CNN** apporte le plus gros gain (+14 points) : les convolutions repèrent des motifs locaux (yeux, bouche, sourcils) où qu'ils soient dans l'image, alors que le réseau dense voit juste une liste de pixels.
- Le **Dropout** règle le gros problème du CNN de base, le surapprentissage : +5.5 points, et une loss presque divisée par 3 (des probabilités beaucoup plus fiables).
- Les autres pistes testées n'ont pas fait mieux : un kernel 5×5 (même accuracy, plus lent, recommence à sur-apprendre) ou s'arrêter plus tôt (évite le surapprentissage mais plafonne plus bas).

### Ce qui marche, ce qui ne marche pas

- **Bien reconnues** : happy (F1 ~0.80) et surprise (~0.72), des expressions très marquées.
- **Difficiles** : fear, sad et neutral, souvent confondues entre elles, et disgust, qui n'a que 436 images d'entraînement et se fait souvent prendre pour angry.
- Une partie des erreurs vient du **dataset** lui-même : des labels discutables, des dessins, des filigranes.
- En conditions réelles (photos, webcam), le système fonctionne en direct sur un simple CPU, mais il est moins fiable sur les visages de profil, mal éclairés ou loin de la caméra, très différents des visages de face de FER-2013.

### Pour aller plus loin

- **Rééquilibrer les classes** (pondérer la loss avec `class_weight`) pour aider disgust et fear.
- **Entraîner un peu plus longtemps** le modèle final : ses courbes montaient encore à l'époque 20.
- **Data Augmentation** (retournements, petites rotations) pour mieux résister aux variations de pose et d'éclairage avec la webcam.